# Cleaning and grounding emerging science concepts: the concept / not-concept classifier

This notebook is a runnable demo of the **concept-cleaning experiment** (artifact `art_BdBvbNuNU8E7`). The full
pipeline (`method.py`) runs 16 steps in order. It trains three models: a concept classifier, a variant merger and
a NIL-aware linker. It applies them to a frozen frame of 426 emerging-concept candidates and then freezes the
test population for the next iteration.

`method.py` itself only drives the pipeline. It calls `src/<step>.py` in order:

```
s0_load → s1a_fetch_arxiv → s1_corpus → t1_reused_code → s2_features → s3_classifier → s4_merger → s4b_merger_choice
→ s5_link_calib → s7a_predict → s6_llm_audit → s7b_merge_link → s6_llm_audit --links → s8_freeze → s9_report → t3_leakage_checks
```

**This demo covers the core of the pipeline: steps `s2_features` and `s3_classifier`.** They contain the pre-registered
**PRIMARY concept classifier**:

* **Features:** lexical (spaCy POS patterns, stop-word share, word frequency and similar), outcome-blind corpus
  **termhood** statistics, **PCA(MiniLM)** and **PCA(SPECTER2)** phrase embeddings.
* **Model:** `StandardScaler` followed by an **L2 logistic regression**. It is tuned by 5-fold `StratifiedGroupKFold`
  (groups are D2 variant clusters) on the train split only.
* **Thresholds:** frozen from out-of-fold (OOF) probabilities *before* the test set is scored. `t_F1` is the argmax of
  OOF F1. `t_P90` is the smallest threshold with OOF precision ≥ 0.90.
* **Evaluation:** ablations, a gradient-boosting comparison, and baselines (majority class, C-value threshold, LLM
  labellers A and B). Metrics carry bootstrap 95% CIs and paired bootstrap differences. The notebook also draws a
  reliability curve and fits the secondary 3-class model.

**Data.** `mini_demo_data.json` holds **100 D2 phrases** (70 train / 30 test) with LLM-adjudicated **silver** labels
(`CONCEPT` / `NOT_CONCEPT` / `TOO_GENERIC`). The full run used 1,200 train and 277 test items. Each phrase carries its
**precomputed termhood statistics**. Step `s1_corpus` computed them with an Aho-Corasick pass over 1.48M arXiv titles
and the DS4 abstracts, which is too large to repeat in a notebook. Lexical features and both embeddings are
**recomputed here** with the original code.

On the full run, the PRIMARY model scored D2-test F1 **0.818** [0.772, 0.859] and AUC **0.888**. Numbers from 100
phrases are much noisier. The final cell sets them next to the full-run reference values.

Steps not reproduced here, because each needs its own large dataset or paid APIs: the variant merger (s4), the
linker calibration (s5), frame application (s7a/s7b), the LLM audit (s6), population freezing (s8), the D4a-augmented
secondary model, and the SemEval/SciERC human anchors.

## 1. Install dependencies
Packages not on Colab are always installed: `loguru`, `wordfreq`, `sentence-transformers` and the spaCy
`en_core_web_sm` model. Colab's pre-installed core packages are only installed off Colab, at Colab's exact versions.

*Note:* the original run loaded SPECTER2 through the `adapters` package, which needs `transformers~=4.57` and would
break Colab's `transformers 5`. The original `Encoder` code already catches this case (**FALLBACK F2**) and falls back
to `specter2_base` with CLS pooling. That is what runs here.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('torch==2.9.0', '--index-url', 'https://download.pytorch.org/whl/cpu')
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0',
         'pyarrow==18.1.0', 'spacy==3.8.11', 'transformers==5.0.0', 'tokenizers==0.22.2', 'huggingface_hub==1.4.0',
         'tqdm==4.67.3', 'requests==2.32.4')

# Packages NOT pre-installed on Colab (always install everywhere)
_pip('loguru==0.7.3', 'wordfreq==3.1.1', 'sentence-transformers==5.7.0')
_pip('--no-deps', 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl')

## 2. Imports
This cell merges the import blocks of `src/common.py`, `featlib.py`, `encoders.py`, `clfdata.py`, `s2_features.py` and
`s3_classifier.py`.

In [ ]:
from __future__ import annotations

# --- common.py
import gzip
import hashlib
import json
import math
import os
import re
import resource
import sys
from collections import defaultdict
from pathlib import Path

from loguru import logger

# --- textnorm.py (vendored)
import unicodedata

# --- featlib.py / clfdata.py / s2_features.py / s3_classifier.py
import itertools
import pickle
import random
import time
from collections import Counter

import joblib
import matplotlib
import numpy as np
import pandas as pd
import torch
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, brier_score_loss, cohen_kappa_score,
                             roc_auc_score, confusion_matrix, f1_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt  # notebook: inline figures (the script used the Agg backend)

## 3. Load the demo data
The data comes from the GitHub URL, with a fallback to the local `mini_demo_data.json`.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-8892b7-occupancy-is-not-integration-for-new/fork/run_F1tk5OGtH84L/round-2/experiment-1/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["metadata"]["description"])
print("label counts:", data["metadata"]["counts"])
print("example:", {k: data["examples"][0][k] for k in ("key", "output", "metadata_fold", "metadata_macro_domain")})

## 4. Configuration
These are all the tunable parameters. The demo keeps grids and bootstrap sizes small enough to finish in a few
minutes on a Colab CPU. The comments give the original full-run values. PCA dimensions are lowered because the
demo's CV training folds have only ~56 rows, and `PCA(n_components)` cannot exceed the number of rows.

In [ ]:
# ======================= CONFIG (demo values; the original full-run values are in the comments) =======================
N_TRAIN = 70          # D2 train phrases used (demo max 70; original: all 1,200 D2-train items)
N_TEST = 30           # D2 test phrases used  (demo max 30; original: all 277 resolved D2-test items)
C_GRID = [0.01]                        # original: [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
CW_GRID = [None]                       # original: [None, "balanced"]
PCA_GRID = [16]                        # original: [64, None]  (PCA(64) needs >= 64 rows per CV training fold)
N_SPLITS = 5                           # original: 5 (StratifiedGroupKFold)
N_BOOT = 50                            # original: 2000 bootstrap replicates
HGB_PCA = 16                           # original: 64
HGB_GRID = {"learning_rate": [0.1], "max_leaf_nodes": [15], "l2_regularization": [0.0]}  # original: [0.05,0.1] x [15,31] x [0.0,1.0]
HGB_MAX_ITER = 50                      # original: 300
RUN_ABLATIONS = ["no_termhood"]        # original: all of ABLATIONS below
ENC_BATCH = 64                         # encoder batch size (original: 512 on GPU)

## 5. Shared helpers (`src/common.py` and `src/vendor/textnorm.py`)
These are the original paths, seed, JSON I/O, hashing, logging and phrase normalisation. Only the notebook-context
changes are marked. Paths point to local `work/`, `models/`, `results/` and `logs/` folders. The mini-run switch is
off, the dependency-workspace loaders are not needed, and `set_limits` is not called because it sets the original
container's RAM cap.

In [ ]:
# ------------------------------------------------------------------ textnorm.py (vendored; only the parts s2/s3 use)
GREEK = {"α": "alpha", "β": "beta", "γ": "gamma", "δ": "delta", "ε": "epsilon", "ζ": "zeta", "η": "eta",
         "θ": "theta", "κ": "kappa", "λ": "lambda", "μ": "mu", "ν": "nu", "ξ": "xi", "π": "pi", "ρ": "rho",
         "σ": "sigma", "τ": "tau", "φ": "phi", "χ": "chi", "ψ": "psi", "ω": "omega", "Δ": "delta",
         "Γ": "gamma", "Ω": "omega", "Φ": "phi", "Ψ": "psi", "Σ": "sigma", "Λ": "lambda", "Θ": "theta"}
BRIT = {"tumour": "tumor", "behaviour": "behavior", "colour": "color", "haemoglobin": "hemoglobin",
        "anaemia": "anemia", "oestrogen": "estrogen", "paediatric": "pediatric", "optimisation": "optimization",
        "modelling": "modeling", "characterisation": "characterization", "organisation": "organization",
        "localisation": "localization", "stabilisation": "stabilization", "polarisation": "polarization",
        "haemorrhage": "hemorrhage", "leukaemia": "leukemia", "oesophageal": "esophageal", "fibre": "fiber",
        "centre": "center", "aluminium": "aluminum", "sulphur": "sulfur", "sulphate": "sulfate",
        "analyse": "analyze", "ageing": "aging", "labour": "labor", "neighbour": "neighbor",
        "haematopoietic": "hematopoietic", "anaesthesia": "anesthesia", "ischaemic": "ischemic",
        "oedema": "edema", "foetal": "fetal", "caesarean": "cesarean", "programme": "program"}
HYPHEN_RE = re.compile(r"[‐-―−/_]+|-")
SPACE_RE = re.compile(r"\s+")


def plural_to_singular(w: str) -> str:
    # Tiny rule-based singulariser for the head token (keeps keys deterministic across processes).
    if len(w) <= 3 or not w.isalpha():
        return w
    if w.endswith(("ss", "us", "is", "ics", "sis", "xis")):
        return w
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"
    if w.endswith(("ches", "shes", "xes", "zes", "sses")):
        return w[:-2]
    if w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w


def normalise(surface: str) -> str:
    s = unicodedata.normalize("NFKC", surface)
    for g, name in GREEK.items():
        if g in s:
            s = s.replace(g, f" {name} ")
    s = s.lower()
    s = HYPHEN_RE.sub(" ", s)
    s = s.replace("'s ", " ").replace("’s ", " ")
    s = re.sub(r"[\"“”‘’`,;:!?]", " ", s)
    s = SPACE_RE.sub(" ", s).strip()
    toks = [BRIT.get(t, t) for t in s.split(" ") if t]
    if not toks:
        return ""
    toks[-1] = plural_to_singular(toks[-1])
    return " ".join(toks)


# ------------------------------------------------------------------ common.py
MINI = False  # notebook: the T2 mini-pipeline switch is off (the demo subset is chosen in the config cell instead)
ROOT = Path(".").resolve()          # notebook: was Path(__file__).resolve().parent.parent
WORK = ROOT / "work"                # derived features: termhood, lexical, embeddings (was cache/)
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
LOGS = ROOT / "logs"
for _d in (WORK, MODELS, RESULTS, LOGS, WORK / "emb"):
    _d.mkdir(parents=True, exist_ok=True)
SEED = 20261001
# N_BOOT comes from the config cell (original: 2000)


def setup_logging(name: str) -> None:
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
    logger.add(LOGS / f"{name}.log", rotation="30 MB", level="DEBUG")


def open_guard(path: str | Path) -> Path:
    # Every data file open goes through here; the sealed outcome file must never be read.
    p = Path(path)
    if "SEALED" in str(p):
        raise PermissionError(f"refusing to open sealed outcome file: {p}")
    return p


def read_json(path: str | Path):
    p = open_guard(path)
    if str(p).endswith(".gz"):
        with gzip.open(p, "rt") as f:
            return json.load(f)
    return json.loads(p.read_text())


def write_json(path: str | Path, obj, indent: int | None = 1) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=indent, ensure_ascii=False, default=_json_default))


def _json_default(o):
    import numpy as np
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (set, frozenset)):
        return sorted(o)
    raise TypeError(type(o))


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(open_guard(path), "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

## 6. Phrase-level lexical features and the embedding store (`src/featlib.py`)
`LexicalFeaturizer` turns each normalised phrase key into 45 hand-crafted features: token and character counts,
stop-word share, the POS of the first and last token, head position, an acronym flag, digits and Greek letters,
Zipf word frequency, a generic-head flag, a malformed-key flag, and a one-hot of the 25 most common POS patterns
(fitted on **D2-train only**). `EmbStore` maps a string to a row of a saved, L2-normalised embedding matrix. The
code is unchanged.

In [ ]:
GENERIC_HEADS = set('''
method approach system study analysis model effect technique process result data application performance
problem framework algorithm structure property function behavior mechanism strategy design development
evaluation investigation measurement characterization assessment comparison scheme procedure tool technology
theory concept factor parameter condition feature role impact influence change increase decrease level rate
value number amount type form kind case example group set part area region field domain aspect issue question
task work research paper article review report experiment test trial observation simulation computation
calculation estimation prediction detection identification classification recognition representation
formulation solution implementation optimization control management treatment therapy intervention outcome
response activity interaction relationship relation correlation association difference similarity distribution
pattern trend variation variability dynamics evolution growth formation generation production synthesis
preparation modification improvement enhancement reduction limitation challenge advantage benefit risk cost
quality efficiency accuracy capability ability potential status state phase stage step period time year
condition environment material sample specimen object component element unit device instrument equipment
network model technique protocol platform service product resource information knowledge evidence finding
estimate measure index indicator score metric criterion standard principle rule law equation expression term
'''.split())
PREP = {"ADP"}
DET = {"DET"}
CONJ = {"CCONJ", "SCONJ"}
VERB = {"VERB", "AUX"}
GREEK_RE = re.compile(r"[Ͱ-Ͽ]|\b(alpha|beta|gamma|delta|epsilon|kappa|lambda|sigma|theta|omega|mu|pi|tau|phi|psi|chi)\b")
SYMBOL_RE = re.compile(r"[^\w\s\-/.,()']")
NONLATIN_RE = re.compile(r"[^\x00-\x7fÀ-ɏͰ-Ͽ‐-―]")
BOILER_RE = re.compile(r"©|all rights reserved|elsevier|springer|wiley|periodical|copyright|\blicensee\b|published by")


def is_acronym_token(t: str) -> bool:
    letters = [c for c in t if c.isalpha()]
    if not letters or len(t) > 6:
        return False
    up = sum(c.isupper() for c in letters)
    return up >= 2 or (up >= 1 and len(letters) <= 3 and t[0].isupper() and any(c.isdigit() for c in t))


class LexicalFeaturizer:
    # Fit the POS-pattern vocabulary on D2-train only; then transform any phrase.

    def __init__(self) -> None:
        import spacy
        from spacy.lang.en.stop_words import STOP_WORDS
        from wordfreq import zipf_frequency
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])
        self.stop = STOP_WORDS
        self.zipf = zipf_frequency
        self.patterns: list[str] = []
        self._pos_cache: dict[str, list[str]] = {}

    def pos(self, keys: list[str]) -> None:
        todo = [k for k in dict.fromkeys(keys) if k not in self._pos_cache]
        for k, doc in zip(todo, self.nlp.pipe(todo, batch_size=1024)):
            self._pos_cache[k] = [t.pos_ for t in doc]

    def fit_patterns(self, keys: list[str], top: int = 25) -> None:
        self.pos(keys)
        c = Counter("_".join(self._pos_cache[k]) for k in keys)
        self.patterns = [p for p, _ in c.most_common(top)]

    def names(self) -> list[str]:
        return (["lx_n_tokens", "lx_n_chars", "lx_stop_share", "lx_first_prep", "lx_first_det", "lx_first_conj",
                 "lx_first_verb", "lx_last_prep", "lx_last_det", "lx_last_conj", "lx_last_verb", "lx_head_pos",
                 "lx_acronym", "lx_has_digit", "lx_has_greek_symbol", "lx_zipf_min", "lx_zipf_mean",
                 "lx_generic_head", "lx_key_malformed"] + [f"lx_pos_{p}" for p in self.patterns] + ["lx_pos_OTHER"])

    def transform(self, key: str, surface_forms: list[str] | None = None, acronyms: list[str] | None = None) -> list[float]:
        # key: the normalised phrase key (lower case). surface_forms keep case for the acronym flag.
        if key not in self._pos_cache:
            self.pos([key])
        pos = self._pos_cache[key]
        toks = key.split()
        n = max(1, len(toks))
        stop_share = sum(t in self.stop for t in toks) / n
        f0 = pos[0] if pos else ""
        fl = pos[-1] if pos else ""
        noun_idx = [i for i, p in enumerate(pos) if p in ("NOUN", "PROPN")]
        head_pos = (noun_idx[-1] + 1) / max(1, len(pos)) if noun_idx else 0.0
        sfs = list(surface_forms or []) + [key]
        acro = float(bool(acronyms) or any(is_acronym_token(t) for s in sfs for t in re.split(r"[\s\-/]+", s)))
        zs = [self.zipf(t, "en") for t in toks] or [0.0]
        head = toks[-1] if toks else ""
        malformed = float(bool(key) and (not key[0].isalnum() or bool(NONLATIN_RE.search(key)) or bool(BOILER_RE.search(key))))
        pat = "_".join(pos)
        onehot = [float(pat == p) for p in self.patterns] + [float(pat not in self.patterns)]
        return [float(len(toks)), float(len(key)), stop_share, float(f0 in PREP), float(f0 in DET), float(f0 in CONJ),
                float(f0 in VERB), float(fl in PREP), float(fl in DET), float(fl in CONJ), float(fl in VERB), head_pos,
                acro, float(any(c.isdigit() for c in key)), float(bool(GREEK_RE.search(" ".join(sfs)) or SYMBOL_RE.search(key))),
                float(min(zs)), float(np.mean(zs)), float(head in GENERIC_HEADS), malformed] + onehot


class EmbStore:
    # String -> row lookup into work/emb/{name}.npy (float16, L2-normalised).

    def __init__(self, name: str) -> None:
        self.name = name
        self.strings = json.loads((WORK / "emb" / f"{name}_strings.json").read_text())
        self.idx = {s: i for i, s in enumerate(self.strings)}
        self.mat = np.asarray(np.load(WORK / "emb" / f"{name}.npy"))
        self.extra: dict[str, np.ndarray] = {}

    def get(self, strings: list[str]) -> np.ndarray:
        miss = [s for s in strings if s not in self.idx and s not in self.extra]
        if miss:
            raise KeyError(f"{self.name}: {len(miss)} strings not embedded, e.g. {miss[:3]}")
        return np.stack([self.mat[self.idx[s]].astype(np.float32) if s in self.idx else self.extra[s] for s in strings]) \
            if strings else np.zeros((0, self.mat.shape[1]), dtype=np.float32)

    def has(self, s: str) -> bool:
        return s in self.idx or s in self.extra

    def add(self, strings: list[str], M: np.ndarray) -> None:
        for s, v in zip(strings, M):
            self.extra[s] = v.astype(np.float32)


def emb_text(s: str) -> str:
    # The exact string given to the encoders for a phrase: normalised key (lower case, hyphens as spaces).
    return normalise(s) or s.lower().strip()

## 7. Phrase encoders (`src/encoders.py`)
* **MiniLM** (`sentence-transformers/all-MiniLM-L6-v2`, mean pooling)
* **SPECTER2** (`allenai/specter2_base` plus the proximity adapter, CLS pooling)

Both return L2-normalised vectors. On a GPU they run in fp16, and the batch size halves on out-of-memory. The code is
unchanged. Without the `adapters` package, SPECTER2 takes its built-in **FALLBACK F2** (base model, no adapter).

In [ ]:
MINILM = "sentence-transformers/all-MiniLM-L6-v2"
SPECTER2_BASE = "allenai/specter2_base"
SPECTER2_ADAPTER = "allenai/specter2"
SAPBERT = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Encoder:
    def __init__(self, name: str) -> None:
        self.name = name
        self.note = ""
        if DEVICE.type == "cuda":
            torch.cuda.set_per_process_memory_fraction(0.8)
        if name == "minilm":
            from sentence_transformers import SentenceTransformer
            self.st = SentenceTransformer(MINILM, device=str(DEVICE))
            if DEVICE.type == "cuda":
                self.st.half()
            self.kind = "st"
        elif name == "specter2":
            from transformers import AutoTokenizer
            self.tok = AutoTokenizer.from_pretrained(SPECTER2_BASE)
            try:
                from adapters import AutoAdapterModel
                m = AutoAdapterModel.from_pretrained(SPECTER2_BASE)
                m.load_adapter(SPECTER2_ADAPTER, source="hf", load_as="specter2", set_active=True)
                self.note = "specter2_base + proximity adapter (adapters.AutoAdapterModel), CLS pooling"
            except Exception as e:  # FALLBACK F2: base model without adapter
                logger.warning(f"SPECTER2 adapter failed ({e!r}); FALLBACK F2 base model")
                from transformers import AutoModel
                m = AutoModel.from_pretrained(SPECTER2_BASE)
                self.note = "FALLBACK F2: specter2_base without adapter, CLS pooling"
            self.model = m.to(DEVICE).eval()
            if DEVICE.type == "cuda":
                self.model.half()
            self.kind = "cls"
        elif name == "sapbert":
            from transformers import AutoModel, AutoTokenizer
            self.tok = AutoTokenizer.from_pretrained(SAPBERT)
            self.model = AutoModel.from_pretrained(SAPBERT).to(DEVICE).eval()
            if DEVICE.type == "cuda":
                self.model.half()
            self.kind = "cls"
            self.note = "SapBERT CLS pooling"
        else:
            raise ValueError(name)

    @torch.inference_mode()
    def encode(self, strings: list[str], batch: int = 512) -> np.ndarray:
        t0 = time.time()
        out = np.zeros((len(strings), self.dim()), dtype=np.float32)
        # sort by length for padding efficiency
        order = np.argsort([len(s) for s in strings])
        i = 0
        bs = batch
        while i < len(order):
            idx = order[i:i + bs]
            texts = [strings[j] for j in idx]
            try:
                if self.kind == "st":
                    v = self.st.encode(texts, batch_size=len(texts), convert_to_numpy=True, normalize_embeddings=True,
                                       show_progress_bar=False)
                else:
                    enc = self.tok(texts, padding=True, truncation=True, max_length=64, return_tensors="pt",
                                   return_token_type_ids=False).to(DEVICE)
                    h = self.model(**enc).last_hidden_state[:, 0, :].float()
                    v = torch.nn.functional.normalize(h, dim=-1).cpu().numpy()
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                bs = max(8, bs // 2)
                logger.warning(f"{self.name}: OOM, batch -> {bs}")
                continue
            out[idx] = v
            i += len(idx)
        logger.info(f"{self.name}: encoded {len(strings)} strings in {time.time() - t0:.1f}s")
        return out

    def dim(self) -> int:
        return 384 if self.name == "minilm" else 768

## 8. Feature matrix, model factory and bootstrap metrics (`src/clfdata.py`)
`FeatureSource.build` concatenates four blocks: **lexical**, **termhood** (11 columns, with a `th_missing` flag),
**MiniLM** and **SPECTER2**. It uses the same code path for D2, D4a and frame phrases. `make_pipeline` scales the
tabular blocks and applies `PCA → StandardScaler` to each embedding block inside a `ColumnTransformer`. The PCA is
therefore fitted on training folds only. The metric helpers compute point metrics, percentile bootstrap CIs, paired
bootstrap differences and the two threshold rules.

This is the original code. The one change: `boot_idx` takes its replicate count from the config (`N_BOOT`) instead
of a hard-coded 2000.

In [ ]:
TH_NAMES = ["th_log_f", "th_src_ratio", "th_log_df", "th_cvalue", "th_right_entropy", "th_left_entropy",
            "th_share_right_ext_noun", "th_share_left_ext_adj_noun", "th_max_right_ext_share", "th_right_boundary_share",
            "th_missing"]


class FeatureSource:
    def __init__(self, with_ctx: bool = True) -> None:
        self.lex = pd.read_parquet(WORK / "lexical.parquet")
        self.lex_names = json.loads((WORK / "lexical_meta.json").read_text())["names"]
        self.lex_idx = {(r.ns, r.id): i for i, r in enumerate(self.lex[["ns", "id"]].itertuples(index=False))}
        self.lexM = self.lex[self.lex_names].to_numpy(dtype=np.float32)
        th = pickle.load(open(WORK / "termhood.pkl", "rb"))
        self.th = th["features"]
        self.th_meta = th["meta"]
        self.mini = EmbStore("minilm")
        self.spec = EmbStore("specter2")
        self.ctx = None
        if with_ctx and (WORK / "emb" / "minilm_ctx.npy").exists():
            self.ctx = EmbStore("minilm_ctx")
            self.ctx_map = json.loads((WORK / "emb" / "ctx_map.json").read_text())

    def build(self, items: list[dict]) -> tuple[np.ndarray, dict[str, list[int]], list[str]]:
        # items: {ns, id, th_qid, text, ctx_key}. Returns X, block -> column indices, column names.
        L = np.stack([self.lexM[self.lex_idx[(it["ns"], it["id"])]] for it in items])
        T = np.array([[self.th.get(it["th_qid"], {}).get(n, 0.0) if it["th_qid"] in self.th else (1.0 if n == "th_missing" else 0.0)
                       for n in TH_NAMES] for it in items], dtype=np.float32)
        texts = [emb_text(it["text"]) for it in items]
        M = self.mini.get(texts)
        S = self.spec.get(texts)
        blocks_arr = [L, T, M, S]
        names = list(self.lex_names) + TH_NAMES + [f"mini_{i}" for i in range(M.shape[1])] + [f"spec_{i}" for i in range(S.shape[1])]
        if self.ctx is not None:
            C = np.zeros((len(items), M.shape[1]), dtype=np.float32)
            flag = np.zeros((len(items), 1), dtype=np.float32)
            for i, it in enumerate(items):
                cs = self.ctx_map.get(it.get("ctx_key") or "", [])
                if cs:
                    C[i] = self.ctx.get(cs).mean(0)
                else:
                    C[i] = M[i]
                    flag[i] = 1.0
            blocks_arr += [C, flag]
            names += [f"ctx_{i}" for i in range(C.shape[1])] + ["ctx_missing"]
        X = np.concatenate(blocks_arr, axis=1)
        o = 0
        blocks = {}
        for b, arr in zip(["lex", "th", "mini", "spec", "ctx", "ctxflag"], blocks_arr):
            blocks[b] = list(range(o, o + arr.shape[1]))
            o += arr.shape[1]
        return X, blocks, names


def make_pipeline(blocks: dict[str, list[int]], use: list[str], *, pca: int | None, model: str = "lr", C: float = 1.0,
                  class_weight=None, hgb: dict | None = None, multinomial: bool = False) -> Pipeline:
    tr = []
    tab = [c for b in use if b in ("lex", "th", "ctxflag") for c in blocks[b]]
    if tab:
        tr.append(("tab", StandardScaler(), tab))
    for b in use:
        if b in ("mini", "spec", "ctx"):
            n = pca if pca else None
            if n:
                tr.append((b, Pipeline([("pca", PCA(n_components=n, random_state=SEED)), ("sc", StandardScaler())]), blocks[b]))
            else:
                tr.append((b, StandardScaler(), blocks[b]))
    ct = ColumnTransformer(tr, remainder="drop")
    if model == "lr":
        clf = LogisticRegression(C=C, class_weight=class_weight, max_iter=5000, solver="lbfgs")
    else:
        clf = HistGradientBoostingClassifier(random_state=SEED, **(hgb or {}))
    return Pipeline([("ct", ct), ("clf", clf)])


# ------------------------------------------------------------------ metrics
def _prf(y: np.ndarray, yhat: np.ndarray) -> tuple[float, float, float]:
    tp = float(((y == 1) & (yhat == 1)).sum())
    fp = float(((y == 0) & (yhat == 1)).sum())
    fn = float(((y == 1) & (yhat == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def point_metrics(y: np.ndarray, p: np.ndarray | None, yhat: np.ndarray) -> dict:
    P, R, F = _prf(y, yhat)
    out = {"n": int(len(y)), "n_pos": int(y.sum()), "precision": P, "recall": R, "f1": F,
           "accuracy": float((y == yhat).mean()), "balanced_accuracy": float(balanced_accuracy_score(y, yhat)),
           "kappa": float(cohen_kappa_score(y, yhat)) if len(set(y)) > 1 or len(set(yhat)) > 1 else 0.0}
    if p is not None and len(set(y)) > 1:
        out["auc"] = float(roc_auc_score(y, p))
        out["auprc"] = float(average_precision_score(y, p))
        out["brier"] = float(brier_score_loss(y, np.clip(p, 0, 1)))
    return out


def boot_idx(n: int, reps: int = N_BOOT, seed: int = SEED) -> np.ndarray:  # notebook: reps from config (original 2000)
    return np.random.default_rng(seed).integers(0, n, size=(reps, n))


def boot_metrics(y: np.ndarray, p: np.ndarray | None, yhat: np.ndarray, idx: np.ndarray | None = None,
                 keys=("precision", "recall", "f1", "accuracy", "auc", "auprc", "brier", "balanced_accuracy", "kappa")) -> dict:
    idx = boot_idx(len(y)) if idx is None else idx
    res: dict[str, list[float]] = {k: [] for k in keys}
    def put(k, v):
        if k in res:
            res[k].append(v)
    for b in idx:
        yb, hb = y[b], yhat[b]
        P, R, F = _prf(yb, hb)
        put("precision", P)
        put("recall", R)
        put("f1", F)
        put("accuracy", float((yb == hb).mean()))
        if len(set(yb)) > 1:
            if "balanced_accuracy" in res:
                put("balanced_accuracy", float(balanced_accuracy_score(yb, hb)))
            if "kappa" in res:
                put("kappa", float(cohen_kappa_score(yb, hb)))
            if p is not None:
                pb = p[b]
                if "auc" in res:
                    put("auc", float(roc_auc_score(yb, pb)))
                if "auprc" in res:
                    put("auprc", float(average_precision_score(yb, pb)))
                if "brier" in res:
                    put("brier", float(brier_score_loss(yb, np.clip(pb, 0, 1))))
    pt = point_metrics(y, p, yhat)
    out = {}
    for k, v in res.items():
        if v and k in pt:
            out[k] = {"point": pt[k], "ci95": [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]}
    out["n"] = pt["n"]
    out["n_pos"] = pt["n_pos"]
    return out


def paired_diff(y: np.ndarray, pa: np.ndarray, ha: np.ndarray, pb: np.ndarray | None, hb: np.ndarray,
                idx: np.ndarray | None = None) -> dict:
    # Paired bootstrap of F1 and AUC differences (model a minus model b) on the same resamples.
    idx = boot_idx(len(y)) if idx is None else idx
    df1, dauc = [], []
    for b in idx:
        yb = y[b]
        df1.append(_prf(yb, ha[b])[2] - _prf(yb, hb[b])[2])
        if len(set(yb)) > 1 and pb is not None:
            dauc.append(roc_auc_score(yb, pa[b]) - roc_auc_score(yb, pb[b]))
    out = {"f1_diff": {"point": _prf(y, ha)[2] - _prf(y, hb)[2],
                       "ci95": [float(np.percentile(df1, 2.5)), float(np.percentile(df1, 97.5))],
                       "p_le_0": float(np.mean(np.array(df1) <= 0))}}
    if dauc:
        out["auc_diff"] = {"point": float(roc_auc_score(y, pa) - roc_auc_score(y, pb)),
                           "ci95": [float(np.percentile(dauc, 2.5)), float(np.percentile(dauc, 97.5))],
                           "p_le_0": float(np.mean(np.array(dauc) <= 0))}
    return out


def best_f1_threshold(y: np.ndarray, p: np.ndarray) -> tuple[float, float]:
    best = (0.5, -1.0)
    for t in np.unique(np.round(p, 4)):
        f = _prf(y, (p >= t).astype(int))[2]
        if f > best[1]:
            best = (float(t), f)
    return best


def precision_threshold(y: np.ndarray, p: np.ndarray, target: float) -> float | None:
    # Smallest threshold with precision >= target (on the given out-of-fold scores).
    for t in np.sort(np.unique(np.round(p, 4))):
        yh = (p >= t).astype(int)
        if yh.sum() == 0:
            break
        if _prf(y, yh)[0] >= target:
            return float(t)
    return None

## 9. Precomputed termhood: the input from step `s1_corpus`
Step `s1_corpus` streamed 1.48M arXiv titles and the DS4 D1 abstracts through an Aho-Corasick automaton. For every
candidate phrase it produced **outcome-blind termhood statistics**: frequency, document frequency, C-value,
left/right-context entropies, extension shares and boundary shares. That pass cannot run in a notebook, so the demo
data carries the result for each phrase. This cell writes it back into the `termhood.pkl` layout that
`FeatureSource` reads (`features["d2:<key>"] = {...}`).

In [ ]:
setup_logging("s2_features")
d2_all = data["examples"]  # notebook: replaces read_json(DATA / "d2.json") (a 100-row D2 subset)
th_pickle = {"features": {f"d2:{r['key']}": r["termhood"] for r in d2_all if r.get("termhood") is not None},
             "queries": None, "frame_title_hits": {},
             "meta": {"source": "precomputed by s1_corpus (arXiv titles <= 2018 + DS4 D1), shipped in mini_demo_data.json"}}
pickle.dump(th_pickle, open(WORK / "termhood.pkl", "wb"))
logger.info(f"termhood rows: {len(th_pickle['features'])} / {len(d2_all)} phrases")

## 10. Step `s2_features`: lexical features and embeddings
**(a) Lexical.** The POS-pattern vocabulary is fitted on D2-train keys only. Every phrase is then transformed and the
table is saved to `work/lexical.parquet`.
**(b) Embeddings.** MiniLM and SPECTER2 embed the normalised phrase strings, which are saved as float16 matrices in
`work/emb/`. The original step also embedded D4a, D3, MeSH, frame and vocabulary strings for the later steps. Here
only the D2 strings are embedded (notebook change). The MiniLM *context* embeddings (`phrase [SEP] title`) feed the
secondary `full_plus_context` variant and are also skipped, so that ablation is skipped too, as the original code
does when they are absent.

In [ ]:
t0 = time.time()
d2 = d2_all  # notebook: the demo subset

# ---------------- (a) lexical
lf = LexicalFeaturizer()
train_keys = [r["key"] for r in d2 if r["metadata_fold"] == "train"]
lf.fit_patterns(train_keys)
rows = []
items = [("d2", r["key"], r["key"], r["surface_forms"], r["acronym_short_forms"]) for r in d2]
# (original also added D4a phrases and the 426 frame concepts here)
lf.pos([it[2] for it in items])
names = lf.names()
for ns, ident, key, sfs, acr in items:
    rows.append({"ns": ns, "id": ident, **dict(zip(names, lf.transform(key, sfs, acr)))})
lex = pd.DataFrame(rows)
lex.to_parquet(WORK / "lexical.parquet")
write_json(WORK / "lexical_meta.json", {"names": names, "pos_patterns_fit_on": "D2-train", "patterns": lf.patterns})
logger.info(f"lexical features: {lex.shape} ({time.time() - t0:.0f}s)")

# ---------------- (b) embeddings
S: list[str] = []
S += [emb_text(r["key"]) for r in d2]
# (original also embedded D4a / D3 / MeSH pairs / held-out MeSH / frame / D5 strings and the OpenAlex + MeSH vocabularies)
S = sorted({s for s in S if s})
logger.info(f"{len(S)} distinct strings to embed")

import gc
notes = {}
for name in ("minilm", "specter2"):
    enc = Encoder(name)
    notes[name] = enc.note
    M = enc.encode(S, batch=ENC_BATCH)  # (original first timed a 10k-string sample to extrapolate the full run)
    np.save(WORK / "emb" / f"{name}.npy", M.astype(np.float16))
    (WORK / "emb" / f"{name}_strings.json").write_text(json.dumps(S, ensure_ascii=False))
    del enc, M
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
write_json(WORK / "emb" / "meta.json", {"n_strings": len(S), "n_ctx": 0, "notes": notes, "runtime_s": time.time() - t0})
logger.info(f"encoder notes: {notes}")
logger.info(f"s2 done in {time.time() - t0:.0f}s")

## 11. Step `s3_classifier`: CV helpers and the tuning grids
* `groups_for` keeps every D2 **variant cluster** inside a single CV fold, so near-duplicate phrases cannot leak
  across folds.
* `oof_proba` returns out-of-fold probabilities from `StratifiedGroupKFold`.
* `tune_lr` grid-searches `C × class_weight × PCA` by **mean per-fold OOF F1 at 0.5**. `tune_hgb` does the same for the
  gradient-boosting comparison.

The code is unchanged except that the grids come from the config cell.

In [ ]:
# (grids from the config cell: C_GRID, CW_GRID, PCA_GRID)
FULL = ["lex", "th", "mini", "spec"]
ABLATIONS = {"no_termhood": ["lex", "mini", "spec"], "no_embeddings": ["lex", "th"], "lexical_only": ["lex"],
             "minilm_only": ["mini"], "specter2_only": ["spec"], "full_plus_context": FULL + ["ctx", "ctxflag"]}
ABLATIONS = {k: v for k, v in ABLATIONS.items() if k in RUN_ABLATIONS}  # notebook: subset chosen in config


def groups_for(rows: list[dict]) -> np.ndarray:
    g = []
    for i, r in enumerate(rows):
        c = r.get("metadata_variant_cluster")
        g.append(str(c) if c is not None else f"solo_{i}")
    return np.array(g)


FOLDS: dict = {}


def oof_proba(make, X, y, groups, n_splits=N_SPLITS) -> np.ndarray:
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    p = np.zeros(len(y))
    fold = np.zeros(len(y), dtype=int)
    for k, (tr, va) in enumerate(cv.split(X, y, groups)):
        m = make()
        m.fit(X[tr], y[tr])
        p[va] = m.predict_proba(X[va])[:, 1]
        fold[va] = k
    FOLDS["last"] = fold
    return p


def mean_fold_f1(y, p, thr=0.5) -> float:
    fold = FOLDS["last"]
    return float(np.mean([f1_score(y[fold == k], (p[fold == k] >= thr).astype(int)) for k in np.unique(fold)]))


def tune_lr(X, y, groups, blocks, use, grid_pca=PCA_GRID) -> tuple[dict, list[dict], np.ndarray]:
    results = []
    best = None
    for C, cw, pca in itertools.product(C_GRID, CW_GRID, grid_pca):
        if not any(b in ("mini", "spec", "ctx") for b in use) and pca is not None:
            continue
        mk = lambda C=C, cw=cw, pca=pca: make_pipeline(blocks, use, pca=pca, C=C, class_weight=cw)
        p = oof_proba(mk, X, y, groups)
        f1 = mean_fold_f1(y, p)  # selection criterion: mean OOF F1 over the 5 folds (at 0.5)
        r = {"C": C, "class_weight": cw, "pca": pca, "oof_f1_at_0.5": f1, "oof_best_thr_f1": best_f1_threshold(y, p)[1],
             "oof_auc": float(point_metrics(y, p, (p >= 0.5).astype(int)).get("auc", np.nan))}
        results.append(r)
        if best is None or r["oof_f1_at_0.5"] > best[0]["oof_f1_at_0.5"]:
            best = (r, p)
    if best is None:  # notebook: an embedding-free ablation with a PCA-only grid has no candidate -> fall back to pca=None
        return tune_lr(X, y, groups, blocks, use, grid_pca=[None])
    return best[0], results, best[1]


def tune_hgb(X, y, groups, blocks, use) -> tuple[dict, list[dict], np.ndarray]:
    results, best = [], None
    for lr, leaves, l2 in itertools.product(HGB_GRID["learning_rate"], HGB_GRID["max_leaf_nodes"], HGB_GRID["l2_regularization"]):
        hp = {"learning_rate": lr, "max_leaf_nodes": leaves, "l2_regularization": l2, "max_iter": HGB_MAX_ITER,
              "early_stopping": False}
        mk = lambda hp=hp: make_pipeline(blocks, use, pca=HGB_PCA, model="hgb", hgb=hp)
        p = oof_proba(mk, X, y, groups)
        f1 = mean_fold_f1(y, p)
        r = {**hp, "oof_f1_at_0.5": f1, "oof_best_thr_f1": best_f1_threshold(y, p)[1]}
        results.append(r)
        if best is None or f1 > best[0]["oof_f1_at_0.5"]:
            best = (r, p)
    return best[0], results, best[1]


def model_hash(path) -> str:
    return sha256_file(path)

## 12. Train/test split, leakage check and feature matrices
`UNRESOLVED` rows are dropped. Silver `CONCEPT` becomes y = 1; `NOT_CONCEPT` and `TOO_GENERIC` become y = 0. The
script **asserts that no normalised key or surface form is shared between train and test**. Then `FeatureSource`
builds `X` from the four feature blocks.

In [ ]:
setup_logging("s3_classifier")
t0 = time.time()
tr_rows = [r for r in d2 if r["metadata_fold"] == "train" and r["output"] != "UNRESOLVED"][:N_TRAIN]
te_rows = [r for r in d2 if r["metadata_fold"] == "test" and r["output"] != "UNRESOLVED"][:N_TEST]
n_unres = sum(r["output"] == "UNRESOLVED" for r in d2 if r["metadata_fold"] in ("train", "test"))
logger.info(f"train {len(tr_rows)} {Counter(r['output'] for r in tr_rows)}; test {len(te_rows)} "
            f"{Counter(r['output'] for r in te_rows)}; dropped UNRESOLVED {n_unres}")
# leakage assert: no normalised key shared between train and test
k_tr = {normalise(r["key"]) for r in tr_rows} | {normalise(s) for r in tr_rows for s in r["surface_forms"]}
k_te = {normalise(r["key"]) for r in te_rows} | {normalise(s) for r in te_rows for s in r["surface_forms"]}
overlap = k_tr & k_te
assert len(overlap) == 0 or MINI, f"D2 train/test key overlap: {list(overlap)[:5]}"

fs = FeatureSource()


def items_d2(rows):
    return [{"ns": "d2", "id": r["key"], "th_qid": f"d2:{r['key']}", "text": r["key"], "ctx_key": f"d2:{r['key']}"} for r in rows]


Xtr, blocks, names = fs.build(items_d2(tr_rows))
Xte, _, _ = fs.build(items_d2(te_rows))
ytr = np.array([r["output"] == "CONCEPT" for r in tr_rows], dtype=int)
yte = np.array([r["output"] == "CONCEPT" for r in te_rows], dtype=int)
gtr = groups_for(tr_rows)
logger.info(f"X train {Xtr.shape}, blocks { {k: len(v) for k, v in blocks.items()} }")

## 13. PRIMARY model: tuning, then freezing the thresholds
The grid search runs on **train only**. `t_F1` and `t_P90` come from the OOF probabilities of the best configuration.
The final pipeline is refit on all training rows. Model and thresholds are written to `models/` *before* the test
set is touched.

In [ ]:
# ---------------- PRIMARY tuning (train only)
best, grid, p_oof = tune_lr(Xtr, ytr, gtr, blocks, FULL)
logger.info(f"PRIMARY best config {best} ({time.time() - t0:.0f}s)")
t_f1, oof_f1 = best_f1_threshold(ytr, p_oof)
t_p90 = precision_threshold(ytr, p_oof, 0.90)
primary = make_pipeline(blocks, FULL, pca=best["pca"], C=best["C"], class_weight=best["class_weight"])
primary.fit(Xtr, ytr)
MODELS.mkdir(exist_ok=True)
joblib.dump({"pipeline": primary, "blocks": blocks, "use": FULL, "names": names, "config": best}, MODELS / "concept_lr.joblib")
thresholds = {"t_F1": t_f1, "oof_f1_at_t_F1": oof_f1, "t_P90": t_p90,
              "t_P90_note": "smallest t with OOF precision >= 0.90" if t_p90 is not None else "precision 0.90 never reached; STRICT uses max OOF prob",
              "config": best, "frozen_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
              "model_sha256": model_hash(MODELS / "concept_lr.joblib")}
if t_p90 is None:
    thresholds["t_P90"] = float(np.max(p_oof))
write_json(MODELS / "thresholds.json", thresholds)
logger.info(f"thresholds frozen: {thresholds}")

## 14. Comparisons: ablations and gradient boosting (same CV)
Each ablation drops feature blocks and is re-tuned with the same CV and the same `t_F1` rule. The `HistGradientBoosting`
comparison uses the full feature set. The original also saved the `no_termhood` model and the best CV comparison model
for the sensitivity columns of `s7a_predict`.

In [ ]:
comp: dict = {}
oofs = {"primary": p_oof}
for name, use in ABLATIONS.items():
    if "ctx" in use and fs.ctx is None:
        continue
    b, _, p = tune_lr(Xtr, ytr, gtr, blocks, use)
    tt, _ = best_f1_threshold(ytr, p)
    m = make_pipeline(blocks, use, pca=b["pca"], C=b["C"], class_weight=b["class_weight"]).fit(Xtr, ytr)
    comp[name] = {"config": b, "t_F1": tt, "model": m, "use": use}
    oofs[name] = p
    logger.info(f"ablation {name}: {b}")
hb, hgrid, hp = tune_hgb(Xtr, ytr, gtr, blocks, FULL)
ht, _ = best_f1_threshold(ytr, hp)
hm = make_pipeline(blocks, FULL, pca=HGB_PCA, model="hgb", hgb={k: hb[k] for k in ("learning_rate", "max_leaf_nodes", "l2_regularization", "max_iter", "early_stopping")}).fit(Xtr, ytr)
comp["hgb"] = {"config": hb, "t_F1": ht, "model": hm, "use": FULL}
oofs["hgb"] = hp
if "no_termhood" in comp:
    joblib.dump({"pipeline": comp["no_termhood"]["model"], "blocks": blocks, "use": comp["no_termhood"]["use"],
                 "config": comp["no_termhood"]["config"], "t_F1": comp["no_termhood"]["t_F1"]}, MODELS / "concept_lr_notermhood.joblib")
# best CV comparison model (F4 sensitivity column): highest OOF best-threshold F1 among comparisons
cv_scores = {k: best_f1_threshold(ytr, v)[1] for k, v in oofs.items() if k != "primary" and k != "full_plus_context"}
best_comp = max(cv_scores, key=cv_scores.get)
joblib.dump({"pipeline": comp[best_comp]["model"], "blocks": blocks, "use": comp[best_comp]["use"], "name": best_comp,
             "t_F1": comp[best_comp]["t_F1"]}, MODELS / "concept_best_comparison.joblib")
write_json(MODELS / "thresholds_comparisons.json", {k: {"t_F1": v["t_F1"], "config": v["config"]} for k, v in comp.items()} |
           {"best_cv_comparison": best_comp, "cv_best_thr_f1": cv_scores})
logger.info(f"best CV comparison: {best_comp}  cv_best_thr_f1={ {k: round(v, 3) for k, v in cv_scores.items()} }")

# ---------------- baselines tuned on train OOF
th_idx = names.index("th_cvalue")
cv_tr = Xtr[:, th_idx]
t_cv, _ = best_f1_threshold(ytr, cv_tr)

## 15. TEST: scored once, after everything above is frozen
The PRIMARY model is scored at `t_F1` and `t_P90`, with bootstrap CIs. Every comparison and baseline gets a **paired**
bootstrap difference (primary minus other) on the same resamples. Baselines: majority class, a threshold on the corpus
**C-value**, and the two LLM labellers. LLM A is `gemini-2.5-flash-lite` and LLM B is `gpt-4.1-nano`. Both helped
create the silver labels, so their scores are **optimistic (circular)**.

In [ ]:
p_te = primary.predict_proba(Xte)[:, 1]
h_te = (p_te >= t_f1).astype(int)
h_te_strict = (p_te >= thresholds["t_P90"]).astype(int)
idx = boot_idx(len(yte))
res: dict = {"n_train": len(ytr), "n_test": len(yte), "train_label_counts": dict(Counter(r["output"] for r in tr_rows)),
             "test_label_counts": dict(Counter(r["output"] for r in te_rows)), "dropped_unresolved": n_unres,
             "train_test_key_overlap": len(overlap), "primary_config": best, "thresholds": thresholds,
             "cv_grid_primary": grid, "hgb_grid": hgrid}
res["test_primary_tF1"] = boot_metrics(yte, p_te, h_te, idx)
res["test_primary_tP90"] = boot_metrics(yte, p_te, h_te_strict, idx)
res["confusion_primary_tF1"] = confusion_matrix(yte, h_te).tolist()
# comparisons
res["test_comparisons"] = {}
for name, c in comp.items():
    pc = c["model"].predict_proba(Xte)[:, 1]
    hc = (pc >= c["t_F1"]).astype(int)
    res["test_comparisons"][name] = {"metrics": boot_metrics(yte, pc, hc, idx), "config": c["config"],
                                     "paired_primary_minus_this": paired_diff(yte, p_te, h_te, pc, hc, idx)}
# baselines
maj = int(ytr.mean() >= 0.5)
base = {"majority_class": (np.full(len(yte), 0.5), np.full(len(yte), maj)),
        "cvalue_threshold": (Xte[:, th_idx], (Xte[:, th_idx] >= t_cv).astype(int)),
        "llm_label_A": (None, np.array([r.get("metadata_label_A") == "CONCEPT" for r in te_rows], dtype=int)),
        "llm_label_B": (None, np.array([r.get("metadata_label_B") == "CONCEPT" for r in te_rows], dtype=int))}
res["test_baselines"] = {}
for name, (pb, hb_) in base.items():
    score = pb if pb is not None else hb_.astype(float)
    m = boot_metrics(yte, score if name != "majority_class" else None, hb_, idx)
    if name == "majority_class":
        m["auc"] = {"point": 0.5, "ci95": [0.5, 0.5]}
    res["test_baselines"][name] = {"metrics": m, "paired_primary_minus_this": paired_diff(yte, p_te, h_te, score, hb_, idx)}
res["test_baselines"]["cvalue_threshold"]["t"] = t_cv
res["baseline_note"] = ("LLM labels A (gemini-2.5-flash-lite) and B (gpt-4.1-nano) helped create the silver labels "
                        "(consensus or haiku adjudication of A!=B), so the A/B baselines are optimistic/circular.")
logger.info(f"TEST primary: F1 {res['test_primary_tF1']['f1']} AUC {res['test_primary_tF1'].get('auc')}")

## 16. Reliability curve, per-domain breakdown and the secondary 3-class model
* The **reliability curve** plots mean predicted P(CONCEPT) against the observed CONCEPT share in 10 uniform bins.
* **Per-macro-domain** metrics are reported for every domain with at least 5 test items.
* The **secondary model** predicts `CONCEPT` / `NOT_CONCEPT` / `TOO_GENERIC` with the primary features and
  `class_weight="balanced"`. The plan called it 4-class, but VARIANT_OF has almost no items, so it has 3 classes.

At the end, the results are written to `results/classifier_results.json` and `results/d2_test_predictions.json`.

In [ ]:
# reliability curve
fr, mp_ = calibration_curve(yte, p_te, n_bins=10, strategy="uniform")
res["reliability_curve"] = {"mean_predicted": mp_.tolist(), "fraction_positive": fr.tolist()}
plt.figure(figsize=(4.2, 4))
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.plot(mp_, fr, "o-", color="#1f77b4", label="primary LR (D2 test)")
plt.xlabel("mean predicted P(CONCEPT)")
plt.ylabel("observed CONCEPT share")
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS / "fig_reliability.png", dpi=150)
plt.show()  # notebook: was plt.close()
# per macro-domain + silver_gold_200
res["per_macro_domain"] = {}
for dom in sorted({r.get("metadata_macro_domain") for r in te_rows}, key=str):
    ii = np.array([r.get("metadata_macro_domain") == dom for r in te_rows])
    if ii.sum() >= 5:
        res["per_macro_domain"][str(dom)] = point_metrics(yte[ii], p_te[ii] if len(set(yte[ii])) > 1 else None, h_te[ii])
sg = np.array([bool(r.get("metadata_silver_gold_200")) for r in te_rows])
if sg.sum():
    y_adj = np.array([r.get("metadata_label_adj") == "CONCEPT" for r in te_rows], dtype=int)
    res["silver_gold_200"] = {"vs_final_silver": point_metrics(yte[sg], p_te[sg], h_te[sg]),
                              "vs_adjudicator_label": point_metrics(y_adj[sg], p_te[sg], h_te[sg]),
                              "llmA_vs_adjudicator_label": point_metrics(
                                  y_adj[sg], None, np.array([r.get("metadata_label_A") == "CONCEPT" for r in te_rows], dtype=int)[sg])}
# ---------------- SECONDARY 4-class
lab3 = {"CONCEPT": 0, "NOT_CONCEPT": 1, "TOO_GENERIC": 2}
y3tr = np.array([lab3.get(r["output"], 1) for r in tr_rows])
y3te = np.array([lab3.get(r["output"], 1) for r in te_rows])
m3 = make_pipeline(blocks, FULL, pca=best["pca"], C=best["C"], class_weight="balanced").fit(Xtr, y3tr)
pr3 = m3.predict(Xte)
res["secondary_3class"] = {"macro_f1": float(f1_score(y3te, pr3, average="macro")),
                           "per_class_f1": dict(zip(lab3, f1_score(y3te, pr3, average=None, labels=[0, 1, 2]).tolist())),
                           "confusion": confusion_matrix(y3te, pr3, labels=[0, 1, 2]).tolist(), "labels": list(lab3),
                           "note": "the plan names this '4-class'; VARIANT_OF has 2 train items and 0 test items, so 3 classes"}
# (original continued with the D2+D4a-augmented secondary model and the SemEval-2017 / SciERC human anchors E1/E2,
#  which need the DS4 D4a dataset and are not part of this demo)
res["runtime_s"] = time.time() - t0
res["feature_names"] = names[:len(blocks["lex"]) + len(blocks["th"])]
write_json(RESULTS / "classifier_results.json", res)
# per-item test predictions (for method_out.json)
write_json(RESULTS / "d2_test_predictions.json", [
    {"key": r["key"], "silver": r["output"], "y": int(y), "p": float(p), "accept_tF1": int(h), "accept_tP90": int(hs),
     "label_A": r.get("metadata_label_A"), "label_B": r.get("metadata_label_B"), "macro_domain": r.get("metadata_macro_domain")}
    for r, y, p, h, hs in zip(te_rows, yte, p_te, h_te, h_te_strict)], indent=None)
logger.info(f"s3 done in {time.time() - t0:.0f}s")

## 17. Results
The table lists every model and baseline on the demo test split (F1 and AUC with bootstrap 95% CIs) next to the
**full-run reference** values (1,200 train / 277 test). The plots show F1 for the demo and the full run, and the
spread of predicted P(CONCEPT) by silver label. With only 30 test phrases the CIs are wide and the ranking can
differ from the full run. The demo shows that the pipeline runs end to end; it does not reproduce the headline
numbers.

In [ ]:
ref = data["reference_full_run"]
rows_ = [("PRIMARY LR (t_F1)", res["test_primary_tF1"], ref["test_primary_tF1"]["f1"]),
         ("PRIMARY LR (t_P90, strict)", res["test_primary_tP90"], None)]
rows_ += [(f"comparison: {k}", v["metrics"], ref["test_comparisons_f1"].get(k)) for k, v in res["test_comparisons"].items()]
rows_ += [(f"baseline: {k}", v["metrics"], ref["test_baselines_f1"].get(k)) for k, v in res["test_baselines"].items()]


def fmt(m, k):
    if k not in m:
        return "   –"
    lo, hi = m[k]["ci95"]
    return f"{m[k]['point']:.3f} [{lo:.2f}, {hi:.2f}]"


print(f"Demo: n_train={res['n_train']}  n_test={res['n_test']}  PRIMARY config={res['primary_config']}")
print(f"Frozen thresholds: t_F1={thresholds['t_F1']:.4f}  t_P90={thresholds['t_P90']:.4f}  "
      f"(full run: t_F1={ref['thresholds']['t_F1']}, t_P90={ref['thresholds']['t_P90']})\n")
print(f"{'model':<34}{'F1 demo [95% CI]':<24}{'AUC demo [95% CI]':<24}{'F1 full run':>11}")
print("-" * 93)
for name, m, rf in rows_:
    print(f"{name:<34}{fmt(m, 'f1'):<24}{fmt(m, 'auc'):<24}{(f'{rf:.3f}' if rf is not None else '–'):>11}")
print("\nPaired bootstrap F1 difference, PRIMARY minus X (demo):")
for grp in ("test_comparisons", "test_baselines"):
    for k, v in res[grp].items():
        d = v["paired_primary_minus_this"]["f1_diff"]
        print(f"  {k:<20} {d['point']:+.3f}  CI [{d['ci95'][0]:+.3f}, {d['ci95'][1]:+.3f}]  P(diff<=0)={d['p_le_0']:.2f}")
print(f"\nSecondary 3-class macro-F1: demo {res['secondary_3class']['macro_f1']:.3f} | full run {ref['secondary_3class_macro_f1']:.3f}")
print("Confusion (PRIMARY @ t_F1, rows = silver 0/1):", res["confusion_primary_tF1"])

# ---- plots
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
labels = [n for n, _, _ in rows_ if n != "PRIMARY LR (t_P90, strict)"]
demo_f1 = [m["f1"]["point"] for n, m, _ in rows_ if n != "PRIMARY LR (t_P90, strict)"]
full_f1 = [(rf if rf is not None else np.nan) for n, _, rf in rows_ if n != "PRIMARY LR (t_P90, strict)"]
yy = np.arange(len(labels))
ax[0].barh(yy - 0.2, demo_f1, height=0.4, label=f"demo (n_test={res['n_test']})", color="#1f77b4")
ax[0].barh(yy + 0.2, full_f1, height=0.4, label="full run (n_test=277)", color="#bbbbbb")
ax[0].set_yticks(yy)
ax[0].set_yticklabels(labels, fontsize=8)
ax[0].invert_yaxis()
ax[0].set_xlabel("F1 (CONCEPT = positive)")
ax[0].set_xlim(0, 1)
ax[0].legend(fontsize=8, loc="lower right")
ax[0].set_title("D2 test F1: demo vs full run")
for lab, col in (("CONCEPT", "#2ca02c"), ("NOT_CONCEPT", "#d62728"), ("TOO_GENERIC", "#ff7f0e")):
    v = [p for r, p in zip(te_rows, p_te) if r["output"] == lab]
    ax[1].hist(v, bins=np.linspace(0, 1, 11), alpha=0.6, label=f"{lab} (n={len(v)})", color=col)
ax[1].axvline(thresholds["t_F1"], color="k", ls="--", lw=1, label="t_F1")
ax[1].axvline(thresholds["t_P90"], color="k", ls=":", lw=1, label="t_P90")
ax[1].set_xlabel("PRIMARY P(CONCEPT) on D2 test")
ax[1].set_ylabel("phrases")
ax[1].legend(fontsize=8)
ax[1].set_title("Predicted probabilities by silver label")
plt.tight_layout()
plt.show()

# a few test phrases with their scores
print(f"\n{'phrase':<45}{'silver':<13}{'p':>6}  accept")
for r, p, h in sorted(zip(te_rows, p_te, h_te), key=lambda x: -x[1])[:12]:
    print(f"{r['key'][:44]:<45}{r['output']:<13}{p:6.3f}  {'yes' if h else 'no'}")